In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.6 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
model_v8m = YOLO('yolov8m.yaml')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/CVBottles

/content/drive/MyDrive/CVBottles


In [ ]:

train_v8m=model_v8m.train(
    data="/content/drive/MyDrive/CVBottles/BottleDataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    conf=0.001,
    iou=0.5,
    patience=30
)


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=0.001, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/CVBottles/BottleDataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.yaml, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=3

In [ ]:
custom_v8m = YOLO("/content/drive/MyDrive/CVBottles/runs/detect/train3/weights/best.pt")
metrics_v8m = custom_v8m.val(data="/content/drive/MyDrive/CVBottles/BottleDataset/data.yaml", split='test', imgsz=640)
print("--------------------------------------------------------")
print("Precision:- ", metrics_v8m.box.p)
print("Recall:- ", metrics_v8m.box.r)
print("F1-Score:- ", metrics_v8m.box.f1)
print("map@50:- ", metrics_v8m.box.map50)
print("map@0.5-0.95:- ", metrics_v8m.box.map)
print("----------------------------------------------------------")

Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8m summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 0.0±0.0 MB/s, size: 16.7 KB)
val: Scanning /content/drive/MyDrive/CVBottles/BottleDataset/test/labels.cache... 90 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 90/90 25.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 13.3s/it 1:20
                   all         90         90      0.999          1      0.995      0.974
Speed: 3.0ms preprocess, 24.3ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /content/drive/MyDrive/CVBottles/runs/detect/val2
--------------------------------------------------------
Precision:-  [     0.9994]
Recall:-  [          1]
F1-Score:-  [     0.9997]
map@50:-  0.995
map@0.5-0.95:-  0.9739199173788406
------------------------------------------------

In [ ]:
from ultralytics import YOLO
import cv2

# -----------------------------
# LOAD MODEL
# -----------------------------
model = YOLO("/content/drive/MyDrive/CVBottles/runs/detect/train3/weights/best.pt")

# -----------------------------
# VIDEO IO
# -----------------------------
cap = cv2.VideoCapture("/content/drive/MyDrive/CVBottles/Bottles.mp4")

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(
    "/content/drive/MyDrive/CVBottles/Bottles4_track_count.mp4",
    fourcc,
    int(cap.get(cv2.CAP_PROP_FPS)),
    (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
     int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
)

# -----------------------------
# COUNTING SETUP
# -----------------------------
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
count_line_x = int(frame_width * 0.40)  # <<< MOVE LINE INTO BOTTLE PATH

counted_ids = set()
prev_positions = {}
total_count = 0

# -----------------------------
# MAIN LOOP
# -----------------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(
        frame,
        persist=True,
        tracker="botsort.yaml",
        conf=0.4,
        imgsz=640,
        classes=[0]  # bottle class
    )

    annotated_frame = results[0].plot()

    # -----------------------------
    # DRAW VERTICAL COUNT LINE
    # -----------------------------
    cv2.line(
        annotated_frame,
        (count_line_x, 0),
        (count_line_x, annotated_frame.shape[0]),
        (0, 0, 255),
        2
    )

    # -----------------------------
    # COUNT LOGIC
    # -----------------------------
    if results[0].boxes.id is not None:
        boxes = results[0].boxes

        for box, track_id in zip(boxes.xyxy, boxes.id):
            x1, y1, x2, y2 = map(int, box)
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            track_id = int(track_id)

            # DEBUG: draw center point
            cv2.circle(annotated_frame, (center_x, center_y), 4, (255, 0, 0), -1)

            prev_x = prev_positions.get(track_id, center_x)
            prev_positions[track_id] = center_x

            # LEFT → RIGHT crossing
            if prev_x > count_line_x and center_x <= count_line_x:

                if track_id not in counted_ids:
                    counted_ids.add(track_id)
                    total_count += 1

    # -----------------------------
    # DISPLAY COUNT
    # -----------------------------
    cv2.putText(
        annotated_frame,
        f"Count: {total_count}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 0),
        3
    )

    out.write(annotated_frame)

# -----------------------------
# CLEANUP
# -----------------------------
cap.release()
out.release()

print("✅ Tracking & counting finished successfully")
print("Total bottles counted:", total_count)



0: 384x640 (no detections), 959.5ms
Speed: 3.0ms preprocess, 959.5ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 948.5ms
Speed: 3.8ms preprocess, 948.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 1216.4ms
Speed: 3.8ms preprocess, 1216.4ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 2374.9ms
Speed: 6.5ms preprocess, 2374.9ms inference, 11.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 2392.1ms
Speed: 14.3ms preprocess, 2392.1ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 964.5ms
Speed: 3.1ms preprocess, 964.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 975.7ms
Speed: 3.2ms preprocess, 975.7ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 963.4ms
Speed: 3.5